### 1. Imports

In [1]:
import os
import requests
from typing import TypedDict

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

### 2. Configuration

#### OpenAI Key

In [2]:
with open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-10\key-vault\openai\api.key") as f:
    openai_api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = openai_api_key

#### URL for CIS Project

In [3]:
CIS_URL = r"http://127.0.0.1:8000/ask"

#### NVD API Key

In [5]:
with open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-9\key-vault\nvd-database\api.key") as f:
    nvd_api_key = f.read().strip()
NVD_API_KEY = nvd_api_key

#### LLM Configuration

In [6]:
MODEL = "gpt-4.1-mini"

In [7]:
llm = ChatOpenAI(model=MODEL, temperature=0)

### 3. Tools Layer

#### CIS Tool

In [10]:
import requests

In [11]:

@tool
def ask_cis_policy(query: str,
                   endpoint: str = "http://127.0.0.1:8000/ask") -> str:
    """
    Calls the CIS Policy Intelligence API and returns only the answer.
    Args:
        query: User query.
        endpoint: FastAPI endpoint URL.
    Returns:
        Answer string.
    Raises:
        requests.HTTPError: If the API returns an error.
        ValueError: If the response does not contain an 'answer' field.
    """
    response = requests.post(
        endpoint,
        json={"query": query},
        timeout=60
    )
    response.raise_for_status()
    data = response.json()
    if "answer" not in data:
        raise ValueError("Response does not contain 'answer'.")
    return data["answer"]

In [12]:
answer = ask_cis_policy.invoke("What are the password policies?")
print(answer)

1. **Answer**  
The password policies recommended in the CIS document include several critical settings aimed at enhancing security. The main recommendations are:  
   - **Enforce Password History**: Set to 24 or more passwords to prevent reuse.  
   - **Maximum Password Age**: Set to 365 or fewer days, but not 0, to ensure regular password changes.  
   - **Minimum Password Age**: Set to 1 or more days to prevent immediate password changes.  
   - **Minimum Password Length**: Set to 14 or more characters to enhance password strength.  
   - **Password Complexity Requirements**: Enabled to ensure passwords include a mix of character types.  
   - **Relax Minimum Password Length Limits**: Enabled for flexibility in certain scenarios.  
   - **Store Passwords Using Reversible Encryption**: Disabled to enhance security.

2. **Relevant CIS Controls**  
- **5.2 Use Unique Passwords**: Use unique passwords for all enterprise assets.  
- **Password Policy Settings**: Ensure compliance with re

#### NVD Tool

In [13]:
@tool
def lookup_cve(keyword:str)->str:
    """Query the NVD API for CVEs matching a keyword."""
    print("[TOOL]CVE Tool Activated")
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {"keywordSearch": keyword, "resultsPerPage": 3}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        data = r.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return "No CVEs found."
        lines = []
        for v in vulns:
            c = v["cve"]
            lines.append(f"{c['id']}: {c['descriptions'][0]['value'][:180]}")
        return "\n".join(lines)
    except Exception as e:
        return f"CVE lookup failed: {e}"

In [17]:
lookup_cve.invoke("SMB")

[TOOL]CVE Tool Activated


'CVE-1999-1387: Windows NT 4.0 SP2 allows remote attackers to cause a denial of service (crash), possibly via malformed inputs or packets, such as those generated by a Linux smbmount command that \nCVE-1999-0225: Windows NT 4.0 allows remote attackers to cause a denial of service via a malformed SMB logon request in which the actual data size does not match the specified size.\nCVE-1999-0495: A remote attacker can gain access to a file system using ..  (dot dot) when accessing SMB shares.'

#### IPCONFIG Tool

In [18]:
import subprocess

@tool
def ipconfig_tool() -> str:
    """
    Returns the Windows network configuration using the ipconfig command.
    """
    try:
        result = subprocess.run(
            ["ipconfig"],
            capture_output=True,
            text=True,
            check=True,
            shell=True
        )

        return result.stdout

    except subprocess.CalledProcessError as e:
        return f"Error executing ipconfig:\n{e.stderr}"

In [ ]:
print(ipconfig_tool.invoke({}))

### 3. Agents Layer

In [21]:
planner = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a planner. Decide weather RAG and CVE lookup is required"
)

In [22]:
retrieval_agent = create_agent(
    model=llm,
    tools=[ask_cis_policy],
    system_prompt="Use the CIS policy tool to retrieve CIS benchmark guidance. Use the provided tools mandatorily"
)

In [23]:
threat_agent = create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="Use CVE lookup tool when threat intelligence is needed. Use the provided tools mandatorily"
)

In [24]:
validator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Validate the evidence and produce a concise final answer"
)

NOTE: you can test the agents independently using <agent>.invoke({})

### 4. State

In [25]:
class CyberState(TypedDict):
    query: str
    plan: str
    rag: str
    cves: str
    final: str

### 5. Nodes

In [26]:
def planner_node(state: CyberState):
    r = planner.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"plan": str(r)}

In [27]:
def rag_node(state: CyberState):
    r = retrieval_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"rag": str(r)}

In [28]:
def cve_node(state: CyberState):
    r = threat_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"cves": str(r)}

In [29]:
def validator_node(state: CyberState):

    prompt = f"""
User Query:
{state["query"]}

Plan:
{state.get("plan", "")}

RAG:
{state.get("rag", "")}

CVEs:
{state.get("cves", "")}

SCORE:
validation score

Produce the final validated response.
Also, give a score from 0 to 10
"""
    r = validator_agent.invoke({
        "messages":[
            {"role":"user", "content":prompt}
        ]
    })
    return {"final": r}

#### 6. Workflow

In [30]:
graph = StateGraph(CyberState)

graph.add_node("planner", planner_node)
graph.add_node("rag", rag_node)
graph.add_node("cve", cve_node)
graph.add_node("validator", validator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "rag")
graph.add_edge("rag", "cve")
graph.add_edge("cve", "validator")
graph.add_edge("validator", END)

In [31]:
app = graph.compile()

### 7. Execute 

In [32]:
result = app.invoke({
    "query": "How can I harden Windows SMB services against ransomware?"
})

In [33]:
result

{'query': 'How can I harden Windows SMB services against ransomware?',
 'plan': "{'messages': [HumanMessage(content='How can I harden Windows SMB services against ransomware?', additional_kwargs={}, response_metadata={}, id='dfdb0c7d-bf78-46b5-9eba-efcd52979395'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be useful here to provide the most up-to-date and specific security measures and known vulnerabilities related to Windows SMB services.\\n\\nI will proceed with RAG and CVE lookup to gather detailed and current information on hardening Windows SMB services against ransomware.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 37, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 

In [34]:
print(result["final"]["messages"][-1].content)

Final Answer:

To harden Windows SMB services against ransomware, implement the following key security measures:

1. **Disable SMBv1**: SMBv1 is outdated and vulnerable. Disable it via Group Policy at  
   `Computer Configuration\Policies\Administrative Templates\MS Security Guide\Configure SMB v1 server` → set to **Disabled**.

2. **Enable SMB Encryption**: Protect SMB traffic by enforcing encryption. Configure Group Policy at  
   `Computer Configuration\Policies\Administrative Templates\Network\Lanman Workstation\Require Encryption` → set to **Enabled**.

3. **Audit Unencrypted SMB Traffic**: Enable auditing to monitor and log SMB communications that do not support encryption. Set these Group Policy paths to **Enabled**:  
   - `Computer Configuration\Policies\Administrative Templates\Network\Lanman Server\Audit client does not support encryption`  
   - `Computer Configuration\Policies\Administrative Templates\Network\Lanman Workstation\Audit server does not support encryption`

4.

In [35]:
query = "Recommend CIS settings for Windows RDP and include recent vulnerabilities that administrators should patch."
result = app.invoke({
    "query": query
})

[TOOL]CVE Tool Activated
[TOOL]CVE Tool Activated


In [36]:
result

{'query': 'Recommend CIS settings for Windows RDP and include recent vulnerabilities that administrators should patch.',
 'plan': "{'messages': [HumanMessage(content='Recommend CIS settings for Windows RDP and include recent vulnerabilities that administrators should patch.', additional_kwargs={}, response_metadata={}, id='fc7b975a-24ef-4aa9-b4e9-a002a4792ebd'), AIMessage(content='For securing Windows Remote Desktop Protocol (RDP), it is important to follow CIS (Center for Internet Security) benchmarks and also be aware of recent vulnerabilities to patch.\\n\\n1. CIS Settings for Windows RDP:\\n- Enable Network Level Authentication (NLA) to require user authentication before establishing a session.\\n- Limit RDP access to specific IP addresses or VPNs using firewall rules.\\n- Set the RDP encryption level to High.\\n- Configure account lockout policies to prevent brute force attacks.\\n- Disable RDP if not needed.\\n- Use strong passwords and multi-factor authentication for RDP users.\

In [37]:
print(result["final"]["messages"][-1].content)

Final validated response:

**CIS Recommended Settings for Windows Remote Desktop Protocol (RDP):**

1. **Enable Network Level Authentication (NLA):** Require user authentication before establishing an RDP session.  
   - Group Policy path:  
     `Computer Configuration\Policies\Administrative Templates\Windows Components\Remote Desktop Services\Remote Desktop Session Host\Security\Require user authentication for remote connections by using Network Level Authentication` → Enabled.

2. **Use strong encryption:** Set RDP encryption level to High to protect session data.

3. **Limit RDP access:** Restrict RDP connections to specific IP addresses or VPNs using firewall rules.

4. **Restrict user access:** Only allow necessary users/groups to log in via RDP.

5. **Configure account lockout policies:** Prevent brute force attacks by locking accounts after repeated failed login attempts.

6. **Disable unnecessary features:** Turn off RemoteFX USB redirection, clipboard redirection, and drive 